# Part 1 - Data Collection

## 1. Project Aim

This project explores how *Alice’s Adventures in Wonderland* can be transformed into a data-driven spatial system.

Three datasets are used:
- Text (narrative structure)
- Images (visual form)
- Audio (temporal behaviour)

This notebook focuses only on collecting raw data.

## 2. Data Sources

### Text
- Local Alice txt file

### Images
- Wikimedia Commons (public illustrations)

### Audio
- Soundscape sources (Pixabay / Freesound)

## 3. Setup

In [62]:
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
import os
from urllib.parse import urlparse

BASE_DIR = Path.cwd().resolve().parent.parent

TEXT_DIR = BASE_DIR / "data" / "raw" / "text"
IMAGE_DIR = BASE_DIR / "data" / "raw" / "images"
AUDIO_DIR = BASE_DIR / "data" / "raw" / "audio"

TEXT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)

BASE_DIR: D:\Work\Workspace\Projects\Python\data-driven-surface


## 4. Text Dataset

The text dataset is created from a local txt file of *Alice’s Adventures in Wonderland*.

The text is segmented into paragraphs for later analysis.

In [63]:
txt_path = TEXT_DIR / "alice_original.txt"

text = txt_path.read_text(encoding="utf-8", errors="ignore")
text = text.replace("\r\n", "\n").replace("\r", "\n")

# 可选：去掉 Gutenberg 头尾
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"

start_idx = text.find(start_marker)
end_idx = text.find(end_marker)

if start_idx != -1 and end_idx != -1:
    text = text[start_idx:end_idx]

# 按章节切分
chapters = re.split(r"CHAPTER\s+[IVXLC]+\.*", text, flags=re.IGNORECASE)
chapters = [c.strip() for c in chapters if c.strip()]

def chunk_text_by_length(text, min_len=300, max_len=800):
    """
    把一整段文本切成较稳定的文本块：
    - 优先按句号、问号、感叹号累积
    - 每块控制在 300~800 字符左右
    """
    # 先清理多余空白
    text = re.sub(r"\s+", " ", text).strip()

    # 按句子粗切
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    current = ""

    for sentence in sentences:
        # 如果加上当前句子还不超过 max_len，就继续累积
        if len(current) + len(sentence) + 1 <= max_len:
            current = f"{current} {sentence}".strip()
        else:
            # 当前块够长才存
            if len(current) >= min_len:
                chunks.append(current)
                current = sentence
            else:
                # 当前块太短，就强行继续拼
                current = f"{current} {sentence}".strip()

    if current:
        chunks.append(current)

    return chunks

rows = []

for chapter_idx, chapter_text in enumerate(chapters, start=1):
    chunks = chunk_text_by_length(chapter_text, min_len=300, max_len=800)

    for chunk_idx, chunk in enumerate(chunks, start=1):
        rows.append({
            "id": f"alice_ch{chapter_idx:02d}_c{chunk_idx:03d}",
            "chapter": chapter_idx,
            "chunk": chunk_idx,
            "text": chunk
        })

text_df = pd.DataFrame(rows)

print("Text rows:", len(text_df))
print(repr(text[:500]))
print("real newline count:", text.count("\n"))
print("literal \\n count:", text.count("\\n"))

text_df.head()

Text rows: 210
"\n                ALICE'S ADVENTURES IN WONDERLAND\n                          Lewis Carroll\n\n                            CHAPTER I\n                      Down the Rabbit-Hole\n\n  Alice was beginning to get very tired of sitting by her sister\non the bank, and of having nothing to do:  once or twice she had\npeeped into the book her sister was reading, but it had no\npictures or conversations in it, `and what is the use of a book,'\nthought Alice `without pictures or conversation?'\n  So she was consideri"
real newline count: 2763
literal \n count: 0


,id,chapter,chunk,text
0,alice_ch01_c001,1,1,ALICE'S ADVENTURES IN WONDERLAND Lewis Carroll
1,alice_ch02_c001,2,1,Down the Rabbit-Hole Alice was beginning to ge...
2,alice_ch02_c002,2,2,I shall be late!' (when she thought it over af...
3,alice_ch02_c003,2,3,The rabbit-hole went straight on like a tunnel...
4,alice_ch02_c004,2,4,She took down a jar from one of the shelves as...


In [64]:
text_df.to_csv(TEXT_DIR / "text_raw.csv", index=False, encoding="utf-8-sig")
print("Saved:", TEXT_DIR / "text_raw.csv")

Saved: D:\Work\Workspace\Projects\Python\data-driven-surface\data\raw\text\text_raw.csv


text_df.to_csv(TEXT_DIR / "text_raw.csv", index=False, encoding="utf-8-sig")

## 5. Image Dataset

The image dataset represents the visual dimension of the project.

To ensure a more stable and reproducible workflow, the image dataset is collected through an official image API rather than HTML page scraping.

The Pexels API is used to retrieve images related to the themes of:
- Alice in Wonderland
- fantasy illustration
- surreal environment

These images will later be used for:
- colour analysis
- visual feature extraction
- image embeddings
- mapping visual properties to 3D form

### 5.1 Set up the API request

In [ ]:
PEXELS_API_KEY = "API_KEY_HERE"  # 替换为你的 Pexels API Key

PEXELS_HEADERS = {
    "Authorization": PEXELS_API_KEY
}

query = "fantasy illustration surreal landscape whimsical art storybook"
per_page = 80
page = 1

pexels_url = f"https://api.pexels.com/v1/search?query={query}&per_page={per_page}&page={page}"
print(pexels_url)

https://api.pexels.com/v1/search?query=fantasy illustration surreal landscape whimsical art storybook&per_page=80&page=1


### 5.2 Request image metadata

In [66]:
response = requests.get(pexels_url, headers=PEXELS_HEADERS, timeout=30)
response.raise_for_status()

pexels_data = response.json()

print("Top-level keys:", pexels_data.keys())
print("Number of photos returned:", len(pexels_data.get("photos", [])))

Top-level keys: dict_keys(['page', 'per_page', 'photos', 'total_results', 'next_page'])
Number of photos returned: 80


In [67]:
photos = pexels_data.get("photos", [])

print("Total photos:", len(photos))

# 看第一条结构
photos[0]

Total photos: 80


{'id': 27776534,
 'width': 4000,
 'height': 6000,
 'url': 'https://www.pexels.com/photo/a-house-on-top-of-a-rock-with-a-tree-27776534/',
 'photographer': 'Ville Aalto',
 'photographer_url': 'https://www.pexels.com/@valolla',
 'photographer_id': 3765074,
 'avg_color': '#82837E',
 'src': {'original': 'https://images.pexels.com/photos/27776534/pexels-photo-27776534.jpeg',
  'large2x': 'https://images.pexels.com/photos/27776534/pexels-photo-27776534.jpeg?auto=compress&cs=tinysrgb&dpr=2&h=650&w=940',
  'large': 'https://images.pexels.com/photos/27776534/pexels-photo-27776534.jpeg?auto=compress&cs=tinysrgb&h=650&w=940',
  'medium': 'https://images.pexels.com/photos/27776534/pexels-photo-27776534.jpeg?auto=compress&cs=tinysrgb&h=350',
  'small': 'https://images.pexels.com/photos/27776534/pexels-photo-27776534.jpeg?auto=compress&cs=tinysrgb&h=130',
  'portrait': 'https://images.pexels.com/photos/27776534/pexels-photo-27776534.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=1200&w=800',
  'landscape'

In [68]:
image_rows = []

for i, photo in enumerate(photos):
    src = photo.get("src", {})
    
    image_rows.append({
        "id": f"img_{i:04d}",
        "source": "pexels",
        "pexels_id": photo.get("id"),
        "width": photo.get("width"),
        "height": photo.get("height"),
        "image_url": src.get("large2x") or src.get("large")
    })

image_meta_df = pd.DataFrame(image_rows)

print("Rows:", len(image_meta_df))
image_meta_df.head()

Rows: 80


,id,source,pexels_id,width,height,image_url
0,img_0000,pexels,27776534,4000,6000,https://images.pexels.com/photos/27776534/pexe...
1,img_0001,pexels,7391802,4069,2713,https://images.pexels.com/photos/7391802/pexel...
2,img_0002,pexels,11592804,2160,2700,https://images.pexels.com/photos/11592804/pexe...
3,img_0003,pexels,4835500,3456,5184,https://images.pexels.com/photos/4835500/pexel...
4,img_0004,pexels,36902049,4032,3024,https://images.pexels.com/photos/36902049/pexe...


### 5.3 Extract image metadata

The returned API records contain image dimensions, IDs, and multiple downloadable image sizes.

A simplified metadata table is created for later downloading and analysis.

In [69]:
image_rows = []

for i, photo in enumerate(photos):
    src = photo.get("src", {})

    image_rows.append({
        "id": f"img_{i:04d}",
        "source": "pexels",
        "pexels_id": photo.get("id"),
        "width": photo.get("width"),
        "height": photo.get("height"),
        "photographer": photo.get("photographer"),
        "description": photo.get("alt"),
        "image_url": src.get("large2x") or src.get("large") or src.get("medium")
    })

image_meta_df = pd.DataFrame(image_rows)

print("Metadata rows:", len(image_meta_df))
image_meta_df.head()

Metadata rows: 80


,id,source,pexels_id,width,height,photographer,description,image_url
0,img_0000,pexels,27776534,4000,6000,Ville Aalto,"A unique house atop a rocky cliff with a tree,...",https://images.pexels.com/photos/27776534/pexe...
1,img_0001,pexels,7391802,4069,2713,Rex Joshua Alarcon America,"Fairytale-inspired castle nestled in Batangas,...",https://images.pexels.com/photos/7391802/pexel...
2,img_0002,pexels,11592804,2160,2700,Mo Eid,A digital surreal landscape featuring pink clo...,https://images.pexels.com/photos/11592804/pexe...
3,img_0003,pexels,4835500,3456,5184,raul romagnoli,Creative depiction of a large blue snake along...,https://images.pexels.com/photos/4835500/pexel...
4,img_0004,pexels,36902049,4032,3024,celal keser,Two whimsical wooden houses nestled in a lush ...,https://images.pexels.com/photos/36902049/pexe...


### 5.4 Define a download function

A reusable function is defined to download image files locally.

In [70]:
def download_file(url, save_path):
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        with open(save_path, "wb") as f:
            f.write(response.content)

        return True

    except Exception as e:
        print("Download failed:", url)
        print("Error:", e)
        return False

### 5.5 Download a test batch of images

A small batch is downloaded first to confirm that the workflow is working correctly before expanding the dataset.

In [71]:
downloaded_rows = []

test_batch = image_meta_df.head(20)

for _, row in test_batch.iterrows():
    url = row["image_url"]

    if not url:
        continue

    save_path = IMAGE_DIR / f"{row['id']}.jpg"
    success = download_file(url, save_path)

    if success:
        downloaded_rows.append({
            **row.to_dict(),
            "local_path": str(save_path)
        })

    time.sleep(0.2)

image_df = pd.DataFrame(downloaded_rows)

print("Downloaded test images:", len(image_df))
image_df.head()

Downloaded test images: 20


,id,source,pexels_id,width,height,photographer,description,image_url,local_path
0,img_0000,pexels,27776534,4000,6000,Ville Aalto,"A unique house atop a rocky cliff with a tree,...",https://images.pexels.com/photos/27776534/pexe...,D:\Work\Workspace\Projects\Python\data-driven-...
1,img_0001,pexels,7391802,4069,2713,Rex Joshua Alarcon America,"Fairytale-inspired castle nestled in Batangas,...",https://images.pexels.com/photos/7391802/pexel...,D:\Work\Workspace\Projects\Python\data-driven-...
2,img_0002,pexels,11592804,2160,2700,Mo Eid,A digital surreal landscape featuring pink clo...,https://images.pexels.com/photos/11592804/pexe...,D:\Work\Workspace\Projects\Python\data-driven-...
3,img_0003,pexels,4835500,3456,5184,raul romagnoli,Creative depiction of a large blue snake along...,https://images.pexels.com/photos/4835500/pexel...,D:\Work\Workspace\Projects\Python\data-driven-...
4,img_0004,pexels,36902049,4032,3024,celal keser,Two whimsical wooden houses nestled in a lush ...,https://images.pexels.com/photos/36902049/pexe...,D:\Work\Workspace\Projects\Python\data-driven-...


### 5.6 Save image metadata

In [72]:
image_df.to_csv(IMAGE_DIR / "image_raw.csv", index=False, encoding="utf-8-sig")
print("Saved:", IMAGE_DIR / "image_raw.csv")

Saved: D:\Work\Workspace\Projects\Python\data-driven-surface\data\raw\images\image_raw.csv


### 5.7 Expand the image dataset

After confirming that the test batch works correctly, the image dataset is expanded by collecting multiple pages from the API.

This makes it possible to build a larger dataset for later analysis and machine learning.

In [73]:
query = "fantasy illustration surreal landscape whimsical art storybook"
per_page = 80
pages_to_collect = 3   # 80 × 3 = 240

### 5.8 Collect metadata across multiple pages

In [74]:
all_photos = []

for page in range(1, pages_to_collect + 1):
    pexels_url = f"https://api.pexels.com/v1/search?query={query}&per_page={per_page}&page={page}"
    
    response = requests.get(pexels_url, headers=PEXELS_HEADERS, timeout=30)
    response.raise_for_status()
    
    page_data = response.json()
    photos = page_data.get("photos", [])
    
    print(f"Page {page}: {len(photos)} photos")
    all_photos.extend(photos)

print("Total photos collected:", len(all_photos))

Page 1: 80 photos
Page 2: 80 photos
Page 3: 80 photos
Total photos collected: 240


### 5.9 Convert metadata into a structured table

In [75]:
image_rows = []

for i, photo in enumerate(all_photos):
    src = photo.get("src", {})

    image_rows.append({
        "id": f"img_{i:04d}",
        "source": "pexels",
        "pexels_id": photo.get("id"),
        "width": photo.get("width"),
        "height": photo.get("height"),
        "photographer": photo.get("photographer"),
        "description": photo.get("alt"),
        "image_url": src.get("large2x") or src.get("large") or src.get("medium")
    })

image_meta_df = pd.DataFrame(image_rows)

# 去重：防止不同页有重复
image_meta_df = image_meta_df.drop_duplicates(subset=["pexels_id"]).reset_index(drop=True)

# 重新生成连续 id
image_meta_df["id"] = [f"img_{i:04d}" for i in range(len(image_meta_df))]

print("Metadata rows after deduplication:", len(image_meta_df))
image_meta_df.head()

Metadata rows after deduplication: 231


,id,source,pexels_id,width,height,photographer,description,image_url
0,img_0000,pexels,27776534,4000,6000,Ville Aalto,"A unique house atop a rocky cliff with a tree,...",https://images.pexels.com/photos/27776534/pexe...
1,img_0001,pexels,7391802,4069,2713,Rex Joshua Alarcon America,"Fairytale-inspired castle nestled in Batangas,...",https://images.pexels.com/photos/7391802/pexel...
2,img_0002,pexels,11592804,2160,2700,Mo Eid,A digital surreal landscape featuring pink clo...,https://images.pexels.com/photos/11592804/pexe...
3,img_0003,pexels,4835500,3456,5184,raul romagnoli,Creative depiction of a large blue snake along...,https://images.pexels.com/photos/4835500/pexel...
4,img_0004,pexels,36902049,4032,3024,celal keser,Two whimsical wooden houses nestled in a lush ...,https://images.pexels.com/photos/36902049/pexe...


### 5.10 Download the expanded batch of images

In [76]:
downloaded_rows = []

for _, row in image_meta_df.iterrows():
    url = row["image_url"]

    if not url:
        continue

    save_path = IMAGE_DIR / f"{row['id']}.jpg"

    # 如果文件已经存在，就不重复下载
    if save_path.exists():
        downloaded_rows.append({
            **row.to_dict(),
            "local_path": str(save_path)
        })
        continue

    success = download_file(url, save_path)

    if success:
        downloaded_rows.append({
            **row.to_dict(),
            "local_path": str(save_path)
        })

    time.sleep(0.2)

image_df = pd.DataFrame(downloaded_rows)

print("Downloaded images:", len(image_df))
image_df.head()

Downloaded images: 231


,id,source,pexels_id,width,height,photographer,description,image_url,local_path
0,img_0000,pexels,27776534,4000,6000,Ville Aalto,"A unique house atop a rocky cliff with a tree,...",https://images.pexels.com/photos/27776534/pexe...,D:\Work\Workspace\Projects\Python\data-driven-...
1,img_0001,pexels,7391802,4069,2713,Rex Joshua Alarcon America,"Fairytale-inspired castle nestled in Batangas,...",https://images.pexels.com/photos/7391802/pexel...,D:\Work\Workspace\Projects\Python\data-driven-...
2,img_0002,pexels,11592804,2160,2700,Mo Eid,A digital surreal landscape featuring pink clo...,https://images.pexels.com/photos/11592804/pexe...,D:\Work\Workspace\Projects\Python\data-driven-...
3,img_0003,pexels,4835500,3456,5184,raul romagnoli,Creative depiction of a large blue snake along...,https://images.pexels.com/photos/4835500/pexel...,D:\Work\Workspace\Projects\Python\data-driven-...
4,img_0004,pexels,36902049,4032,3024,celal keser,Two whimsical wooden houses nestled in a lush ...,https://images.pexels.com/photos/36902049/pexe...,D:\Work\Workspace\Projects\Python\data-driven-...


### 5.11 Save the expanded image dataset

In [77]:
image_df.to_csv(IMAGE_DIR / "image_raw.csv", index=False, encoding="utf-8-sig")
print("Saved:", IMAGE_DIR / "image_raw.csv")

Saved: D:\Work\Workspace\Projects\Python\data-driven-surface\data\raw\images\image_raw.csv


### 5.12 Observations

The image dataset has now been expanded to a larger collection using multiple API pages.

This provides enough image samples for:
- colour-based analysis
- feature extraction
- image vectorisation
- later clustering and mapping to 3D form

## 6. Audio Dataset

The audio dataset represents the temporal and atmospheric dimension of the project.

To keep the workflow stable and reproducible, the dataset is collected through the official Freesound API.

Instead of spoken narration, the dataset focuses on non-linguistic soundscapes, such as:
- wind
- forest ambience
- ticking sounds
- surreal mechanical textures
- fantasy-like environmental sounds

These sounds will later be used for:
- temporal feature extraction
- spectral analysis
- animation timing
- mapping sound dynamics to 3D behaviour

### 6.1 Set up the Freesound API request

In [78]:
FREESOUND_API_KEY = "KjXeC27hyukA80B5nYF9XHjZIXwP2fAy7Vjkjlkj"

audio_queries = [
    "wind",
    "forest",
    "rain",
    "clock ticking",
    "mechanical",
    "drone",
    "ambience"
]

audio_page_size = 50
audio_pages_per_query = 1

audio_fields = ",".join([
    "id",
    "name",
    "tags",
    "username",
    "license",
    "duration",
    "previews",
    "description"
])

### 6.2 Collect metadata using multiple simple queries

Instead of relying on one long search phrase, the audio dataset is collected through several simpler keyword queries.

This improves search reliability and produces a more diverse soundscape dataset.

In [79]:
all_sounds = []

headers = {
    "Authorization": f"Token {FREESOUND_API_KEY}"
}

for query in audio_queries:
    print(f"\n--- Query: {query} ---")

    for page in range(1, audio_pages_per_query + 1):
        freesound_url = (
            "https://freesound.org/apiv2/search/text/"
            f"?query={query}"
            f"&page={page}"
            f"&page_size={audio_page_size}"
            f"&fields={audio_fields}"
        )

        response = requests.get(freesound_url, headers=headers, timeout=30)
        response.raise_for_status()

        page_data = response.json()
        results = page_data.get("results", [])

        print(f"Page {page}: {len(results)} sounds")
        all_sounds.extend(results)

print("\nTotal sounds collected before deduplication:", len(all_sounds))


--- Query: wind ---
Page 1: 50 sounds

--- Query: forest ---
Page 1: 50 sounds

--- Query: rain ---
Page 1: 50 sounds

--- Query: clock ticking ---
Page 1: 50 sounds

--- Query: mechanical ---
Page 1: 50 sounds

--- Query: drone ---
Page 1: 50 sounds

--- Query: ambience ---
Page 1: 50 sounds

Total sounds collected before deduplication: 350


### 6.3 Convert sound metadata into a structured table

In [80]:
audio_rows = []

for i, sound in enumerate(all_sounds):
    previews = sound.get("previews", {}) or {}

    preview_url = (
        previews.get("preview-hq-mp3")
        or previews.get("preview-lq-mp3")
        or previews.get("preview-hq-ogg")
        or previews.get("preview-lq-ogg")
    )

    audio_rows.append({
        "id": f"audio_{i:04d}",
        "source": "freesound",
        "freesound_id": sound.get("id"),
        "name": sound.get("name"),
        "username": sound.get("username"),
        "license": sound.get("license"),
        "duration": sound.get("duration"),
        "tags": ", ".join(sound.get("tags", [])) if sound.get("tags") else "",
        "description": sound.get("description"),
        "audio_url": preview_url
    })

audio_meta_df = pd.DataFrame(audio_rows)

audio_meta_df = audio_meta_df.drop_duplicates(subset=["freesound_id"]).reset_index(drop=True)
audio_meta_df["id"] = [f"audio_{i:04d}" for i in range(len(audio_meta_df))]

print("Metadata rows after deduplication:", len(audio_meta_df))
print("Rows with audio_url:", audio_meta_df["audio_url"].notna().sum())

audio_meta_df.head()

Metadata rows after deduplication: 350
Rows with audio_url: 350


,id,source,freesound_id,name,username,license,duration,tags,description,audio_url
0,audio_0000,freesound,383184,chattering teeth - winding up,ChrisReierson,https://creativecommons.org/licenses/by/4.0/,9.77104,"chattering, effect, funny, humor, mechanical, ...",Winding up a miniature chattering teeth toy. I...,https://cdn.freesound.org/previews/383/383184_...
1,audio_0001,freesound,383169,wind gap indoors 002 170305_1100.wav,klankbeeld,https://creativecommons.org/licenses/by/4.0/,64.16610,"air-flow, air-stream, ajar, ambience, ambient,...",The wind ajar trough a door-gap indoors (10cm ...,https://cdn.freesound.org/previews/383/383169_...
2,audio_0002,freesound,383139,wind gap indoors 001 170305_1100.wav,klankbeeld,https://creativecommons.org/licenses/by/4.0/,110.95900,"air-flow, air-stream, ajar, ambience, ambient,...",The wind ajar trough a door-gap indoors (10cm ...,https://cdn.freesound.org/previews/383/383139_...
3,audio_0003,freesound,379470,Wind3.mp3,vandale,http://creativecommons.org/publicdomain/zero/1.0/,5.01406,"air, breeze, gust, swoosh, wind",wind breeze swoosh air gust,https://cdn.freesound.org/previews/379/379470_...
4,audio_0004,freesound,379467,Wind2.ogg,vandale,http://creativecommons.org/publicdomain/zero/1.0/,3.95977,"air, breeze, gust, swoosh, wind",wind breeze swoosh air gust,https://cdn.freesound.org/previews/379/379467_...


### 6.4 Download audio files

In [81]:
def download_audio_file(url, save_path):
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        with open(save_path, "wb") as f:
            f.write(response.content)

        return True

    except Exception as e:
        print("Audio download failed:", url)
        print("Error:", e)
        return False

In [82]:
test_audio_df = audio_meta_df[audio_meta_df["audio_url"].notna()].head(20).copy()

audio_paths = []

for _, row in test_audio_df.iterrows():
    url = row["audio_url"]

    if ".ogg" in url:
        ext = ".ogg"
    else:
        ext = ".mp3"

    file_path = AUDIO_DIR / f"{row['id']}{ext}"

    try:
        success = download_audio_file(url, file_path)
        if success:
            audio_paths.append(str(file_path))
        else:
            audio_paths.append(None)
    except Exception:
        audio_paths.append(None)

test_audio_df["local_path"] = audio_paths

print("Downloaded audio files:", test_audio_df["local_path"].notna().sum())
test_audio_df.head()

Downloaded audio files: 20


,id,source,freesound_id,name,username,license,duration,tags,description,audio_url,local_path
0,audio_0000,freesound,383184,chattering teeth - winding up,ChrisReierson,https://creativecommons.org/licenses/by/4.0/,9.77104,"chattering, effect, funny, humor, mechanical, ...",Winding up a miniature chattering teeth toy. I...,https://cdn.freesound.org/previews/383/383184_...,D:\Work\Workspace\Projects\Python\data-driven-...
1,audio_0001,freesound,383169,wind gap indoors 002 170305_1100.wav,klankbeeld,https://creativecommons.org/licenses/by/4.0/,64.16610,"air-flow, air-stream, ajar, ambience, ambient,...",The wind ajar trough a door-gap indoors (10cm ...,https://cdn.freesound.org/previews/383/383169_...,D:\Work\Workspace\Projects\Python\data-driven-...
2,audio_0002,freesound,383139,wind gap indoors 001 170305_1100.wav,klankbeeld,https://creativecommons.org/licenses/by/4.0/,110.95900,"air-flow, air-stream, ajar, ambience, ambient,...",The wind ajar trough a door-gap indoors (10cm ...,https://cdn.freesound.org/previews/383/383139_...,D:\Work\Workspace\Projects\Python\data-driven-...
3,audio_0003,freesound,379470,Wind3.mp3,vandale,http://creativecommons.org/publicdomain/zero/1.0/,5.01406,"air, breeze, gust, swoosh, wind",wind breeze swoosh air gust,https://cdn.freesound.org/previews/379/379470_...,D:\Work\Workspace\Projects\Python\data-driven-...
4,audio_0004,freesound,379467,Wind2.ogg,vandale,http://creativecommons.org/publicdomain/zero/1.0/,3.95977,"air, breeze, gust, swoosh, wind",wind breeze swoosh air gust,https://cdn.freesound.org/previews/379/379467_...,D:\Work\Workspace\Projects\Python\data-driven-...


In [83]:
audio_paths = []

for _, row in audio_meta_df.iterrows():
    url = row["audio_url"]

    if not url:
        audio_paths.append(None)
        continue

    if ".ogg" in url:
        ext = ".ogg"
    else:
        ext = ".mp3"

    file_path = AUDIO_DIR / f"{row['id']}{ext}"

    if file_path.exists():
        audio_paths.append(str(file_path))
        continue

    try:
        success = download_audio_file(url, file_path)
        if success:
            audio_paths.append(str(file_path))
        else:
            audio_paths.append(None)
    except Exception:
        audio_paths.append(None)

audio_meta_df["local_path"] = audio_paths

print("Downloaded audio files:", audio_meta_df["local_path"].notna().sum())
audio_meta_df.head()

Downloaded audio files: 350


,id,source,freesound_id,name,username,license,duration,tags,description,audio_url,local_path
0,audio_0000,freesound,383184,chattering teeth - winding up,ChrisReierson,https://creativecommons.org/licenses/by/4.0/,9.77104,"chattering, effect, funny, humor, mechanical, ...",Winding up a miniature chattering teeth toy. I...,https://cdn.freesound.org/previews/383/383184_...,D:\Work\Workspace\Projects\Python\data-driven-...
1,audio_0001,freesound,383169,wind gap indoors 002 170305_1100.wav,klankbeeld,https://creativecommons.org/licenses/by/4.0/,64.16610,"air-flow, air-stream, ajar, ambience, ambient,...",The wind ajar trough a door-gap indoors (10cm ...,https://cdn.freesound.org/previews/383/383169_...,D:\Work\Workspace\Projects\Python\data-driven-...
2,audio_0002,freesound,383139,wind gap indoors 001 170305_1100.wav,klankbeeld,https://creativecommons.org/licenses/by/4.0/,110.95900,"air-flow, air-stream, ajar, ambience, ambient,...",The wind ajar trough a door-gap indoors (10cm ...,https://cdn.freesound.org/previews/383/383139_...,D:\Work\Workspace\Projects\Python\data-driven-...
3,audio_0003,freesound,379470,Wind3.mp3,vandale,http://creativecommons.org/publicdomain/zero/1.0/,5.01406,"air, breeze, gust, swoosh, wind",wind breeze swoosh air gust,https://cdn.freesound.org/previews/379/379470_...,D:\Work\Workspace\Projects\Python\data-driven-...
4,audio_0004,freesound,379467,Wind2.ogg,vandale,http://creativecommons.org/publicdomain/zero/1.0/,3.95977,"air, breeze, gust, swoosh, wind",wind breeze swoosh air gust,https://cdn.freesound.org/previews/379/379467_...,D:\Work\Workspace\Projects\Python\data-driven-...


### 6.6 Save the audio dataset

In [84]:
audio_meta_df.to_csv(AUDIO_DIR / "audio_raw.csv", index=False, encoding="utf-8-sig")
print("Saved:", AUDIO_DIR / "audio_raw.csv")

Saved: D:\Work\Workspace\Projects\Python\data-driven-surface\data\raw\audio\audio_raw.csv


In [85]:
print("Rows with audio_url:", audio_meta_df["audio_url"].notna().sum())
print("Downloaded audio files:", test_audio_df["local_path"].notna().sum())

Rows with audio_url: 350
Downloaded audio files: 20


### 6.7 Download full audio dataset

In [86]:
audio_paths = []

for _, row in audio_meta_df.iterrows():
    url = row["audio_url"]

    if not url:
        audio_paths.append(None)
        continue

    # 自动判断格式
    if ".ogg" in url:
        ext = ".ogg"
    else:
        ext = ".mp3"

    file_path = AUDIO_DIR / f"{row['id']}{ext}"

    # 已存在就跳过（避免重复下载）
    if file_path.exists():
        audio_paths.append(str(file_path))
        continue

    try:
        success = download_audio_file(url, file_path)

        if success:
            audio_paths.append(str(file_path))
        else:
            audio_paths.append(None)

        import time
        time.sleep(0.1)

    except Exception as e:
        print("Error:", e)
        audio_paths.append(None)

# 写回 dataframe
audio_meta_df["local_path"] = audio_paths

print("Downloaded audio files:", audio_meta_df["local_path"].notna().sum())

Downloaded audio files: 350


## Next Step

The next notebook will focus on data cleaning and standardisation.

This includes:
- text normalisation and filtering
- checking image metadata and file consistency
- validating audio files and preparing them for feature extraction